In [2]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Toastmasters_Three_Page_Guide (1).pdf")

documents = loader.load()

print("Total Pages:", len(documents))

/tmp/ipykernel_1001/2228141018.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total Pages: 3


In [4]:
for page in documents:
    print(page.page_content)
    print("="*100)

Toastmasters International
Toastmasters International is a nonprofit educational organization dedicated to helping individuals
develop communication, public speaking, and leadership skills. Established in 1924 by Ralph C.
Smedley, Toastmasters has expanded into a global network of clubs where members learn through
practice, feedback, and continuous improvement.
Many people join Toastmasters to overcome stage fear, become confident speakers, improve their
interview skills, strengthen workplace communication, and develop leadership abilities. Unlike
traditional classroom learning, Toastmasters provides a practical environment where members
actively participate in meetings and learn by doing.
A Toastmasters club creates a supportive atmosphere where every member is encouraged to
speak, learn, and grow. Members progress at their own pace and receive constructive feedback
that helps them identify strengths and areas for improvement. This unique learning approach has
helped millions of peopl

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of Chunks:", len(chunks))

Number of Chunks: 22


In [6]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}")
    print(chunk.page_content)
    print("-"*50)


Chunk 1
Toastmasters International
Toastmasters International is a nonprofit educational organization dedicated to helping individuals
develop communication, public speaking, and leadership skills. Established in 1924 by Ralph C.
--------------------------------------------------

Chunk 2
Smedley, Toastmasters has expanded into a global network of clubs where members learn through
practice, feedback, and continuous improvement.
Many people join Toastmasters to overcome stage fear, become confident speakers, improve their
--------------------------------------------------

Chunk 3
interview skills, strengthen workplace communication, and develop leadership abilities. Unlike
traditional classroom learning, Toastmasters provides a practical environment where members
actively participate in meetings and learn by doing.
--------------------------------------------------


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS Database Created Successfully")

FAISS Database Created Successfully


In [9]:
query = "What is Toastmasters?"

results = db.similarity_search(
    query,
    k=2
)

for doc in results:
    print(doc.page_content)
    print("="*100)

Toastmasters International
Toastmasters International is a nonprofit educational organization dedicated to helping individuals
develop communication, public speaking, and leadership skills. Established in 1924 by Ralph C.
relationships while learning together.
In conclusion, Toastmasters is much more than a speaking club. It is a personal and professional
development organization that empowers individuals to become confident speakers, effective


In [11]:
query = "What are the officer roles in Toastmasters?"

results = db.similarity_search(query)

print(results[0].page_content)

Leadership, Pathways, and Benefits
Toastmasters provides numerous leadership opportunities through club officer roles. Members can
serve as President, Vice President Education, Vice President Membership, Vice President Public
